# Minimum-Variance portofolio

Assume that an investor can create a portfolio by allocating to three assets _A_, _B_ and _C_ with the following correlation matrix $\boldsymbol{C}$, vector of volatilities $\boldsymbol{v}$, and expected return vector $\boldsymbol{\mu}$

$$
\boldsymbol{C} = \begin{pmatrix} 1.0 & 0.2 & 0.15 \\
0.2 & 1.0 & 0.4 \\
0.15 & 0.4 & 1.0\end{pmatrix}
$$

and 

$$
\boldsymbol{v} = \begin{pmatrix} 0.1 \\ 0.15 \\ 0.2 \end{pmatrix}
$$


and 

$$
\boldsymbol{\mu} = \begin{pmatrix} 0.04 \\ 0.06 \\ 0.08 \end{pmatrix}
$$

In [1]:
# we only need the numpy package
import numpy as np

## Problem 1

Define the correlation matrix, covariance matrix and the vector of expected returns as `numpy` arrays

In [2]:
# Korrelationsmatrix
C = np.array([[1.0, 0.2, 0.15],
              [0.2, 1.0, 0.4],
              [0.15, 0.4, 1.0]])

# Volatilitetsvektor
v = np.array([0.1, 0.15, 0.2])

# Kovariansmatrix (std x std x korrelation)
Sigma = np.matmul(np.diag(v), np.matmul(C, np.diag(v)))

# Forventet afkastvektor
mu = np.array([0.04, 0.06, 0.08])

# Print for at tjekke resultatet
print(Sigma)
print(mu)

[[0.01   0.003  0.003 ]
 [0.003  0.0225 0.012 ]
 [0.003  0.012  0.04  ]]
[0.04 0.06 0.08]


**Tolkning:** Diagonalen i Sigma er hver akties egen varians 
(0.1² = 0.01, 0.15² = 0.0225, 0.2² = 0.04). 
De øvrige celler viser kovariansen (samspillet) mellem aktiepar.

## Problem 2

Check if the covariance matrix is postive definite. 

In [3]:
# beregn egenværdierne for Sigma
eigenvalues = np.linalg.eigvals(Sigma)
print(eigenvalues)

# tjek om alle er positive
print(np.all(eigenvalues > 0))

[0.04654728+0.j 0.00931378+0.j 0.01663894+0.j]
True


**Tolkning:** Alle tre egenværdier for Sigma er positive (0.0465, 0.0093, 0.0166), hvilket bekræfter at kovariansmatricen er positiv definit. Det betyder at porteføljevariansen altid vil være et positivt tal, uanset hvilke vægte man vælger — så Sigma er en gyldig kovariansmatrix.

## Problem 3

Consider an equally weighted portfolio $w_A = w_B = w_C = 1/3$. What is the formula for the portfolio variance using matrix notation? Calculate the portfolio standard devation for the equally weighted portfolio. Also calculate the expected return for the portfolio. 


In [4]:
# ligevægtede vægte (1/3 til hver)
w = np.array([1/3, 1/3, 1/3])

# forventet porteføljeafkast
mu_p = np.matmul(w, mu)
print(mu_p)

# porteføljevarians og standardafvigelse
sigma_p2 = np.matmul(w, np.matmul(Sigma, w))
sigma_p = np.sqrt(sigma_p2)
print(sigma_p)

0.06
0.10979779394667069


**Tolkning:** Med en ligevægtet portefølje (1/3 i hver aktie) er det forventede afkast 6%, med en standardafvigelse (risiko) på ca. 10.98%. Formlen for porteføljeafkast er "vægte ganget med afkastvektor" (matmul(w, mu)), og formlen for porteføljevarians er "vægte ganget med kovariansmatrix ganget med vægte" (matmul(w, matmul(Sigma, w))).

## Problem 4

The portfolio weights $\mathbf{w} = (w_A \; \; w_B \; \; w_C)^\top$ satisfies the budget constraint $\mathbf{w}^\top \mathbf{1} =  w_A + w_B + w_C = 1$. The investor seeks to minimize the variance subject to this condition

$$
\min \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w} \; \; \;  \text{s.t. } \mathbf{w}^\top \mathbf{1}  = 1
$$

The Lagrangian reads

$$
\mathcal{L} = \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w} - \lambda (\mathbf{w}^\top \mathbf{1}  - 1)
$$

Deriving the FOC and solving for the optimal weights yields

$$
\mathbf{w}^* = \frac{\boldsymbol{\Sigma}^{-1} \mathbf{1}}{\mathbf{1}^\top \boldsymbol{\Sigma}^{-1} \mathbf{1}}
$$

Define a function which calculates the minimum-variance portfolio weights given a covariance matrix. Calculate the minimum-variance portfolio weights. 


In [6]:
def minimum_variance_weights(Sigma: np.ndarray) -> np.ndarray:
    """
    Calculates the minimum-variance portfolio weights.
    
    Parameters
    ----------
    Sigma: np.ndarray
        Covariance matrix of asset returns.
        
    Returns
    -------
    np.ndarray
        Minimum-variance portfolio weights.
    """
    n = Sigma.shape[0]
    ones = np.ones(n)
    Sigma_inv = np.linalg.inv(Sigma)
    return np.matmul(Sigma_inv, ones) / np.matmul(ones, np.matmul(Sigma_inv, ones))

# beregn vægtene ved at kalde funktionen
w_star = minimum_variance_weights(Sigma)
print(w_star)

[0.70384615 0.21538462 0.08076923]


**Tolkning:** Den minimum-varians-optimerede portefølje allokerer 70.4% til aktie A, 21.5% til B og 8.1% til C, hvilket giver et forventet afkast på 4.75% og en risiko på 8.90%. Sammenlignet med den ligevægtede portefølje (6% afkast, 10.98% risiko) opnås markant lavere risiko, på bekostning af lavere forventet afkast — den klassiske afvejning mellem risiko og afkast.